# Тест гибридного поиска чанков по запросу

Вызывает тот же пайплайн, что и CLI/API: **BM25 (Elasticsearch)** + **GNN** (`stage7_1_retriever`).

**Перед запуском:**
- Запущены Elasticsearch, Qdrant (для legacy GNN), Neo4j, Ollama с моделью из `OLLAMA_MODEL`.
- Если используете эмбеддинги из обучающего ноутбука: `GNN_USE_NEO4J_EMBEDDINGS=true`, в Neo4j записаны `Chunk.gnn_embedding`, создан vector index (`stage7_1_retriever/neo4j/chunk_gnn_embedding_vector_index.cypher`).
- Иначе (legacy): нужен файл `GNN_MODEL_PATH` и коллекция Qdrant `QDRANT_COLLECTION`.
- Переменные можно задать в `stage7_1_retriever/.env` (подхватывается `load_config()`).
- Если `gnn_weights.json` нет, следующая ячейка **автоматически** включит Neo4j-режим (повторный `load_config()`).

In [8]:
from pathlib import Path
import sys

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "stage7_1_retriever").is_dir():
            return base
    raise RuntimeError("Запустите kernel из корня репозитория или из notebooks/")

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

REPO_ROOT = /home/user/Documents/Projects/ML/LLMOtusProject/GNN-Retriever-


In [15]:
import logging
import os
from pathlib import Path

import pandas as pd

from stage7_1_retriever import build_retriever_from_env
from stage7_1_retriever.config import load_config

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s - %(message)s")

config = load_config()

# Legacy GNN требует JSON по GNN_MODEL_PATH. Если файла нет — переключаемся на gnn_embedding из Neo4j.
_wpath = Path(config.gnn.model_path)
if not _wpath.is_absolute():
    _wpath = REPO_ROOT / _wpath
if not config.gnn.use_neo4j_embeddings and not _wpath.is_file():
    os.environ["GNN_USE_NEO4J_EMBEDDINGS"] = "true"
    print(
        "GNN: не найден файл весов — включён Neo4j-режим (gnn_embedding + vector index):\n ",
        _wpath,
        "\n  Для legacy положите gnn_weights.json или задайте GNN_MODEL_PATH.\n",
    )
    config = load_config()

print(
    "GNN_USE_NEO4J_EMBEDDINGS:", config.gnn.use_neo4j_embeddings,
    "| index:", config.gnn.neo4j_gnn_vector_index,
    "| ES:", config.elasticsearch.index_name,
    "| Qdrant:", config.qdrant.collection_name,
)


GNN_USE_NEO4J_EMBEDDINGS: True | index: chunk_gnn_embedding | ES: legal_chunks | Qdrant: legal_chunks_vectors


In [20]:
from neo4j import GraphDatabase

# Выполнить один раз после записи gnn_embedding в Chunk
INDEX_NAME = config.gnn.neo4j_gnn_vector_index
DIM = config.ollama.embedding_dim

cypher = f"""
CREATE VECTOR INDEX {INDEX_NAME} IF NOT EXISTS
FOR (c:Chunk)
ON (c.gnn_embedding)
OPTIONS {{
  indexConfig: {{
    `vector.dimensions`: {int(DIM)},
    `vector.similarity_function`: 'cosine'
  }}
}}
"""

with GraphDatabase.driver(
    config.neo4j.uri,
    auth=(config.neo4j.user, config.neo4j.password),
) as driver:
    with driver.session(database=config.neo4j.database) as session:
        session.run(cypher)
        rows = session.run(
            "SHOW INDEXES YIELD name, type, state WHERE name = $n RETURN name, type, state",
            n=INDEX_NAME,
        )
        rec = rows.single()
        print("CREATE OK; index record:", dict(rec) if rec else "not listed yet (проверьте SHOW INDEXES вручную)")


CREATE OK; index record: {'name': 'chunk_gnn_embedding', 'type': 'VECTOR', 'state': 'POPULATING'}


## Запросы

Измените `QUERIES` и при необходимости `TOP_K` / `FILTERS`. Перед запуском гибрида выполните ячейку **Elasticsearch по chunk_id** (функции `fetch_elasticsearch_by_chunk_ids`, `chunks_to_df_with_es`).

In [10]:
QUERIES = [
    "Who is Lincoln"
]

TOP_K = 5
FILTERS = None  # например {"doc_id": "some-id"} если индекс поддерживает

## Elasticsearch по `chunk_id`

После гибридного ретрива список `chunk_id` дополнительно запрашивается в индексе `config.elasticsearch.index_name` (тот же запрос, что `Retriever._enrich_text`). Функции ниже нужны и для ячейки **GNN-only** — выполните эту ячейку до неё.

In [24]:
from elasticsearch import AsyncElasticsearch


async def fetch_elasticsearch_by_chunk_ids(chunk_ids: list[str]) -> dict[str, dict]:
    """Документы из Elasticsearch по полю `chunk_id` (как в `Retriever._enrich_text`)."""
    if not chunk_ids:
        return {}
    unique = list(dict.fromkeys(chunk_ids))
    es = AsyncElasticsearch(config.elasticsearch.url)
    try:
        resp = await es.search(
            index=config.elasticsearch.index_name,
            body={"size": len(unique), "query": {"terms": {"chunk_id": unique}}},
        )
        out: dict[str, dict] = {}
        for hit in resp.get("hits", {}).get("hits", []):
            src = hit.get("_source") or {}
            cid = str(src.get("chunk_id") or "")
            if cid:
                row = dict(src)
                row["_es_id"] = hit.get("_id")
                row["_es_score"] = hit.get("_score")
                out[cid] = row
        return out
    finally:
        await es.close()


def chunks_to_df(chunks):
    rows = []
    for i, c in enumerate(chunks, start=1):
        text_preview = " ".join((c.text or "").split())[:400]
        rows.append(
            {
                "rank": i,
                "chunk_id": c.chunk_id,
                "score": round(c.score, 6),
                "bm25": c.source_scores.get("bm25"),
                "gnn": c.source_scores.get("gnn"),
                "text_preview": text_preview,
                "metadata": c.metadata,
            }
        )
    return pd.DataFrame(rows)


def chunks_to_df_with_es(chunks, es_by_chunk_id: dict[str, dict], *, text_preview_len: int = 500) -> pd.DataFrame:
    """Склейка скоров ретривера с полями из ES (`text`, `normalized_text`, метаданные)."""
    rows = []
    for i, c in enumerate(chunks, start=1):
        src = es_by_chunk_id.get(c.chunk_id, {})
        text_es = str(src.get("text") or "")
        norm_es = str(src.get("normalized_text") or "")
        primary = text_es or norm_es
        preview = " ".join(primary.split())[:text_preview_len] if primary else " ".join((c.text or "").split())[:text_preview_len]
        rows.append(
            {
                "rank": i,
                "chunk_id": c.chunk_id,
                "in_es": c.chunk_id in es_by_chunk_id,
                "score": round(c.score, 6),
                "bm25": c.source_scores.get("bm25"),
                "gnn": c.source_scores.get("gnn"),
                "doc_id": src.get("doc_id"),
                "version_id": src.get("version_id"),
                "language": src.get("language"),
                "es_text_preview": preview,
                "es_text_len": len(primary),
                "metadata": c.metadata,
            }
        )
    return pd.DataFrame(rows)


In [25]:
async def retrieve_chunks(query: str, *, top_k: int = TOP_K, filters=None):
    retriever = build_retriever_from_env(config)
    try:
        return await retriever.retrieve(query_text=query, filters=filters, top_k=top_k)
    finally:
        await retriever.close()


all_results = {}
for q in QUERIES:
    results = await retrieve_chunks(q, filters=FILTERS)
    es_by_id = await fetch_elasticsearch_by_chunk_ids([c.chunk_id for c in results])
    all_results[q] = {"chunks": results, "elasticsearch": es_by_id}
    n_ids = len({c.chunk_id for c in results})
    print(f"\n=== query: {q!r} -> {len(results)} chunks, документов в ES: {len(es_by_id)}/{n_ids} ===")
    display(chunks_to_df_with_es(results, es_by_id))
    # Полные поля _source по chunk_id: all_results[q]["elasticsearch"][chunk_id]

INFO httpx - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


INFO httpx - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO elastic_transport.transport - POST http://localhost:9200/legal_chunks/_search [status:200 duration:0.082s]
INFO stage7_1_retriever - BM25 retrieval done query_len=14 results=30
WARNING stage7_1_retriever - Neo4j vector index query failed: {neo4j_code: Neo.ClientError.Security.Unauthorized} {message: The client is unauthorized due to authentication failure.} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}
INFO stage7_1_retriever - Neo4j GNN retrieval has no vector-index seeds (check index ONLINE and gnn_embedding)
INFO stage7_1_retriever - Hybrid retrieval done query_len=14 bm25=30 gnn=0 returned=5
INFO elastic_transport.transport - POST http://localhost:9200/legal_chunks/_search [status:200 duration:0.011s]



=== query: 'Who is Lincoln' -> 5 chunks, документов в ES: 5/5 ===


,rank,chunk_id,in_es,score,bm25,gnn,doc_id,version_id,language,es_text_preview,es_text_len,metadata
0,1,019d8288-d219-7660-9d7a-99b22e632ba0,True,0.500000,19.004877,None,307,v1,und,Family and childhood Early life Abraham Lincol...,1206,"{'doc_id': '307', 'version_id': 'v1', 'languag..."
1,2,019d8288-d229-74f1-9797-ea1ea23c0da8,True,0.398196,17.997920,None,307,v1,und,"In 1849, he received a patent for a flotation ...",1134,"{'doc_id': '307', 'version_id': 'v1', 'languag..."
2,3,019d8288-d26f-7980-b3d7-d2687e81aa38,True,0.342542,17.447441,None,307,v1,und,Lincoln became a favorite of liberal intellect...,1198,"{'doc_id': '307', 'version_id': 'v1', 'languag..."
3,4,019d8288-d26d-7873-8333-0c446c2df4da,True,0.342542,17.447441,None,307,v1,und,"Americans asked, ""What would Lincoln do?"" Howe...",1095,"{'doc_id': '307', 'version_id': 'v1', 'languag..."
4,5,019d8288-d254-75eb-b038-82dd620fa8f2,True,0.339996,17.422266,None,307,v1,und,"His main goal was to keep the union together, ...",1118,"{'doc_id': '307', 'version_id': 'v1', 'languag..."


## Опционально: только канал GNN

Сначала выполните ячейку **Elasticsearch по chunk_id** (определения `fetch_elasticsearch_by_chunk_ids` и `chunks_to_df_with_es`).

In [27]:
from stage7_1_retriever.gnn_retriever import GnnRetriever
from elasticsearch import AsyncElasticsearch
from neo4j import AsyncGraphDatabase
from qdrant_client import AsyncQdrantClient


async def gnn_only_retrieve(query: str, *, top_k: int = TOP_K):
    elastic = AsyncElasticsearch(config.elasticsearch.url)
    qdrant = AsyncQdrantClient(url=config.qdrant.url)
    neo4j_driver = AsyncGraphDatabase.driver(
        config.neo4j.uri,
        auth=("neo4j", "password"),
    )
    gnn = GnnRetriever(
        qdrant_client=qdrant,
        qdrant_config=config.qdrant,
        neo4j_driver=neo4j_driver,
        neo4j_config=config.neo4j,
        ollama_config=config.ollama,
        gnn_config=config.gnn,
        default_top_k=top_k,
    )
    try:
        return await gnn.retrieve(query_text=query, top_k=top_k)
    finally:
        await gnn.close()
        await elastic.close()


q = QUERIES[0]
gnn_chunks = await gnn_only_retrieve(q)
es_map = await fetch_elasticsearch_by_chunk_ids([c.chunk_id for c in gnn_chunks])
print(f"GNN-only for {q!r}: {len(gnn_chunks)} results, ES: {len(es_map)} docs")
display(chunks_to_df_with_es(gnn_chunks, es_map))

INFO httpx - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
INFO httpx - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO stage7_1_retriever - Neo4j GNN retrieval done query_len=14 results=5
INFO elastic_transport.transport - POST http://localhost:9200/legal_chunks/_search [status:200 duration:0.006s]


GNN-only for 'Who is Lincoln': 5 results, ES: 5 docs


,rank,chunk_id,in_es,score,bm25,gnn,doc_id,version_id,language,es_text_preview,es_text_len,metadata
0,1,019d8288-d26a-7219-aae7-be281cbc6929,True,0.701921,None,0.701921,307,v1,und,"The term ""the United States"" has historically ...",1220,"{'entity_count': 4, 'neo4j_gnn': True, 'neo4j_..."
1,2,019d8288-d215-76d0-8758-bdf226f203a4,True,0.701682,None,0.701682,307,v1,und,"Abraham Lincoln ( ; February 12, 1809 – April ...",1151,"{'entity_count': 11, 'neo4j_gnn': True, 'neo4j..."
2,3,019d8288-d218-753a-b915-3cb0f86ca9fa,True,0.672296,None,0.672296,307,v1,und,"It also directed the Army and Navy to ""recogni...",1130,"{'entity_count': 5, 'neo4j_gnn': True, 'neo4j_..."
3,4,019d8288-d227-7da1-b743-43c795f2dee7,True,0.657367,None,0.657367,307,v1,und,The war had begun with a killing of American s...,1161,"{'entity_count': 5, 'neo4j_gnn': True, 'neo4j_..."
4,5,019d8288-d26f-7980-b3d7-d2687e81aa38,True,0.656103,None,0.656103,307,v1,und,Lincoln became a favorite of liberal intellect...,1198,"{'entity_count': 6, 'neo4j_gnn': True}"
